In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Import libraries

from src.dependence.dependence import cov_matrix
from src.data.returns import split_returns
from src.dependence.copula_modelling import load_vine_model
from src.data.wrangler import get_kept_tickers
from src.optimisation.hmv.clustering import clustering_matrix, dependence_matrix
from src.optimisation.hmv.seriation import quasi_diagonalisation
from src.optimisation.hmv.recursive_bisection import heuristic_optimisation, gamma_sensitivity, plot_gamma_sensitivity
 
from src.data.returns import * 
from src.optimisation.mvp.mvp_solver import *
from src.optimisation.hrp.hrp_weights import hrp_weights
from src.optimisation.eqw.eq_weights import equal_weights

from src.performance.rolling_weights import (
    rolling_weights,
    plot_rolling_weights,
    plot_turnover_comparison
)

from src.performance.portfolio_metrics import portfolio_metrics, sub_period_metrics

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [ ]:
t_matrix = dependence_matrix(1)
t_matrix.head()

In [ ]:
t_matrix.to_csv('../data/output/results/matrix_t1.csv')


In [ ]:
load_vine_model()

In [ ]:
train, test = split_returns()

In [ ]:
hmv_weights_train = heuristic_optimisation(train)
hmv_returns_test = test @ hmv_weights_train



In [ ]:
hrp_weights_train = hrp_weights(train)
hrp_returns_test = test @ hrp_weights_train


In [ ]:
eqw_weights_train = equal_weights(returns=train)
eqw_returns_test = test @ eqw_weights_train

In [ ]:
mvp_weights_train = mvp_weights(train.cov())
mvp_returns_test = test @ mvp_weights_train

In [ ]:
# Collect all return series into one DataFrame
returns_df = pd.DataFrame({
    'HMV-Copula' : hmv_returns_test,
    'HRP'        : hrp_returns_test,
    'Equal Weight': eqw_returns_test,
    'MVP'        : mvp_returns_test
})

# Confirm shape and check for nulls
print(returns_df.shape)
print(returns_df.isnull().sum())
print(returns_df.describe())

In [ ]:
# Apply to every column uniformly
table1 = pd.DataFrame(
    {col: portfolio_metrics(returns_df[col]) for col in returns_df.columns}
).T

table1.index.name = 'Strategy'
print(table1.to_string())

In [ ]:
# LaTeX (paste directly into your thesis)
print(table1.to_latex(float_format="%.2f", bold_rows=True))

# Excel (easier to format manually)
table1.to_excel('table1_performance.xlsx')

In [ ]:
table2 = sub_period_metrices(returns_df=returns_df)
print(table2.to_string())

In [ ]:
results = gamma_sensitivity(
    train_returns=train,
    test_returns=test,
    gamma_grid=np.linspace(0, 1, 51)
)

In [ ]:
optimal_gamma, optimal_sharpe = plot_gamma_sensitivity(results)

In [ ]:
final_weights = heuristic_optimisation(
    train_returns=train,
    gamma=optimal_gamma
)

In [ ]:
# ── Cell: Figure 4 — Rolling Weight Stability ─────────────────────────────
from src.performance.rolling_weights import (
    rolling_weights,
    plot_rolling_weights,
    plot_turnover_comparison
)
from src.data.returns import get_log_returns

# use full returns for the rolling window — not just the test split
full_returns = get_log_returns()

# compute rolling weights (this takes a few minutes)
weight_dfs = rolling_weights(
    returns_df   = full_returns,
    train_window = 504,          # 2-year training window
    refit_freq   = 63,           # refit every quarter
    gamma        = 0.88           # replace with optimal_gamma from Figure 3
)




In [ ]:
# Figure 4a — stacked area charts
plot_rolling_weights(weight_dfs, top_n=10)

# Figure 4b — turnover bar chart
# plot_turnover_comparison(weight_dfs)

In [ ]:
plot_turnover_comparison(weight_dfs)

In [ ]:
# ── Cell: Figure 6 — Cumulative Returns ───────────────────────────────────
from src.performance.cumulative_returns import (
    plot_cumulative_returns,
    print_crisis_performance
)

# plot Figure 6 — uses returns_df already built earlier in the notebook
plot_cumulative_returns(returns_df, log_scale=True)